# ASIC Pipeline Walkthrough

This notebook mirrors the current ASIC pipeline in two explicit stages:

1. harmonize the ASIC source tables into shared static and dynamic tables,
2. review an overview of the harmonized data and essential QC information,
3. build the Chapter 1 cohort and create the 8-hour block outputs.

The notebook intentionally calls the shared pipeline code from the repository rather than re-implementing the logic locally.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 260)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "icu_data_platform").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/icu_data_platform")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from icu_data_platform.sources.asic.extract.raw_tables import (  # noqa: E402
    DEFAULT_ASIC_RAW_DATA_DIR,
    DEFAULT_TRANSLATION_PATH,
)
from icu_data_platform.sources.asic.pipeline import (  # noqa: E402
    DEFAULT_ASIC_HARMONIZED_OUTPUT_DIR,
    build_asic_chapter1_dataset,
    build_asic_harmonized_dataset,
    write_asic_chapter1_dataset,
    write_asic_harmonized_dataset,
)

PROJECT_ROOT


## Pipeline Parameters


In [ ]:
PIPELINE_PARAMETERS = {
    "raw_dir": DEFAULT_ASIC_RAW_DATA_DIR,
    "translation_path": DEFAULT_TRANSLATION_PATH,
    "min_non_null": 20,
    "min_hospitals": 4,
    "fence_factor": 1.5,
}
OUTPUT_DIR = PROJECT_ROOT / DEFAULT_ASIC_HARMONIZED_OUTPUT_DIR

if not PIPELINE_PARAMETERS["raw_dir"].exists():
    raise FileNotFoundError(f"ASIC raw data directory not found: {PIPELINE_PARAMETERS['raw_dir']}")

display(pd.Series({key: str(value) for key, value in PIPELINE_PARAMETERS.items()}, name="value").to_frame())
print(f"Output directory: {OUTPUT_DIR}")


## 1. Harmonize ASIC Data


In [ ]:
harmonized_dataset = build_asic_harmonized_dataset(**PIPELINE_PARAMETERS)
harmonized_output_paths = write_asic_harmonized_dataset(
    harmonized_dataset,
    output_dir=OUTPUT_DIR,
)

final_static = harmonized_dataset.static.combined.copy()
final_dynamic = harmonized_dataset.dynamic.combined.copy()

harmonized_overview = pd.DataFrame(
    [
        {
            "table": "static",
            "rows": final_static.shape[0],
            "columns": final_static.shape[1],
            "hospitals": final_static["hospital_id"].nunique(dropna=True),
            "unique_stays": final_static["stay_id_global"].nunique(dropna=True),
        },
        {
            "table": "dynamic",
            "rows": final_dynamic.shape[0],
            "columns": final_dynamic.shape[1],
            "hospitals": final_dynamic["hospital_id"].nunique(dropna=True),
            "unique_stays": final_dynamic["stay_id_global"].nunique(dropna=True),
        },
    ]
)

display(Markdown("### Harmonized Output Paths"))
display(pd.Series({key: str(path) for key, path in harmonized_output_paths.items()}, name="path").to_frame())

display(Markdown("### Harmonized Table Overview"))
display(harmonized_overview)


### Harmonized Data Overview and Essential Information


In [ ]:
stay_id_qc_summary = harmonized_dataset.stay_id_qc.summary.copy()

static_rows_by_hospital = (
    final_static.groupby("hospital_id", dropna=False)
    .size()
    .rename("static_rows")
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)

dynamic_time_coverage = (
    final_dynamic.assign(parsed_time=pd.to_timedelta(final_dynamic["time"], errors="coerce"))
    .groupby("hospital_id", dropna=False)
    .agg(
        dynamic_rows=("time", "size"),
        unique_stays=("stay_id_global", lambda s: s.nunique(dropna=True)),
        min_time=("parsed_time", "min"),
        max_time=("parsed_time", "max"),
    )
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)

qc_snapshot = pd.DataFrame(
    [
        {
            "artifact": "stay_id_mapping_failures",
            "rows": harmonized_dataset.stay_id_qc.mapping_failures.shape[0],
        },
        {
            "artifact": "duplicate_dynamic_time_index",
            "rows": harmonized_dataset.stay_id_qc.duplicate_dynamic_time_index.shape[0],
        },
        {
            "artifact": "dynamic_non_numeric_issues",
            "rows": harmonized_dataset.dynamic.non_numeric_issues.shape[0],
        },
        {
            "artifact": "dynamic_distribution_issues",
            "rows": harmonized_dataset.dynamic.distribution_issues.shape[0],
        },
    ]
)

static_preview_columns = [
    column
    for column in ["hospital_id", "stay_id_global", "stay_id_local", "sex", "age_group", "readmission", "icu_mortality"]
    if column in final_static.columns
]
dynamic_preview_columns = [
    column
    for column in ["hospital_id", "stay_id_global", "time", "minutes_since_admit", "heart_rate", "map", "sbp", "dbp", "spo2", "resp_rate"]
    if column in final_dynamic.columns
]

display(Markdown("### Stay-ID QC Summary"))
display(stay_id_qc_summary)

display(Markdown("### Static Rows by Hospital"))
display(static_rows_by_hospital)

display(Markdown("### Dynamic Time Coverage by Hospital"))
display(dynamic_time_coverage)

display(Markdown("### QC Snapshot"))
display(qc_snapshot)

display(Markdown("### Static Preview"))
display(final_static[static_preview_columns].head(10))

display(Markdown("### Dynamic Preview"))
display(final_dynamic[dynamic_preview_columns].head(10))

display(Markdown("### Static Schema Summary"))
display(harmonized_dataset.static.schema_summary)

display(Markdown("### Dynamic Schema Summary"))
display(harmonized_dataset.dynamic.schema_summary)


## 3. Build Chapter 1 Cohort and 8-Hour Blocks


In [ ]:
chapter1_dataset = build_asic_chapter1_dataset(
    harmonized_dataset,
    raw_dir=PIPELINE_PARAMETERS["raw_dir"],
)
chapter1_output_paths = write_asic_chapter1_dataset(
    chapter1_dataset,
    output_dir=OUTPUT_DIR,
)

final_cohort = chapter1_dataset.cohort.table.copy()
chapter1_stays = chapter1_dataset.cohort.chapter1.table.copy()
chapter1_block_index = chapter1_dataset.chapter1_8h_blocks.block_index.copy()
chapter1_blocked_dynamic_features = chapter1_dataset.chapter1_8h_blocks.blocked_dynamic_features.copy()

chapter1_overview = pd.DataFrame(
    [
        {
            "artifact": "authoritative_stay_level_cohort",
            "rows": final_cohort.shape[0],
            "columns": final_cohort.shape[1],
        },
        {
            "artifact": "chapter1_retained_stays",
            "rows": chapter1_stays.shape[0],
            "columns": chapter1_stays.shape[1],
        },
        {
            "artifact": "chapter1_8h_block_index",
            "rows": chapter1_block_index.shape[0],
            "columns": chapter1_block_index.shape[1],
        },
        {
            "artifact": "chapter1_8h_blocked_dynamic_features",
            "rows": chapter1_blocked_dynamic_features.shape[0],
            "columns": chapter1_blocked_dynamic_features.shape[1],
        },
    ]
)

display(Markdown("### Chapter 1 Output Paths"))
display(pd.Series({key: str(path) for key, path in chapter1_output_paths.items()}, name="path").to_frame())

display(Markdown("### Chapter 1 Overview"))
display(chapter1_overview)

display(Markdown("### Retained Hospitals"))
display(chapter1_dataset.cohort.chapter1.retained_hospitals)

display(Markdown("### Chapter 1 Counts by Hospital"))
display(chapter1_dataset.cohort.chapter1.counts_by_hospital)

display(Markdown("### Block QC Summary"))
display(chapter1_dataset.chapter1_8h_blocks.qc_summary)

display(Markdown("### Block Index Preview"))
display(chapter1_block_index.head(10))

display(Markdown("### Blocked Dynamic Feature Preview"))
blocked_preview_columns = [
    column
    for column in [
        "stay_id_global",
        "hospital_id",
        "block_index",
        "block_start_h",
        "block_end_h",
        "dynamic_row_count",
        "non_missing_measurements_in_block",
        "observed_variables_in_block",
        "heart_rate_obs_count",
        "heart_rate_median",
        "heart_rate_last",
        "map_obs_count",
        "map_median",
        "map_last",
        "sbp_obs_count",
        "sbp_median",
        "sbp_last",
        "spo2_obs_count",
        "spo2_median",
        "spo2_last",
    ]
    if column in chapter1_blocked_dynamic_features.columns
]
display(chapter1_blocked_dynamic_features[blocked_preview_columns].head(10))


In [ ]:
chapter1_blocked_dynamic_features

## Working Objects

The main notebook variables are:

- `harmonized_dataset`: harmonized static/dynamic tables plus stay-ID QC
- `final_static`: harmonized static table
- `final_dynamic`: harmonized dynamic table
- `chapter1_dataset`: Chapter 1 cohort plus 8-hour block outputs
- `final_cohort`: authoritative stay-level cohort table
- `chapter1_stays`: retained Chapter 1 stay-level cohort
- `chapter1_block_index`: completed 8-hour block index
- `chapter1_blocked_dynamic_features`: block-level aggregated dynamic feature table
- `harmonized_output_paths`: written static/dynamic/qc artifact paths
- `chapter1_output_paths`: written cohort/block artifact paths
